# Viagem — vyhodnocení parcely

Doplněk k balíčku. Prompt: `prompts/system.txt`, `prompts/user_template.txt`. Runner: `evaluate.py`.


## Úloha

Z dat o jedné parcele strukturované posouzení výkupu: skóre, doporučení, rizika, jistota, `overit_rucne`. Data ručně; výstup dle `schema/vyhodnoceni.schema.json`.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Kořen balíčku = rodič složky notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)
assert (ROOT / "prompts" / "system.txt").is_file(), "spusť notebook z kořene balíčku nebo z notebooks/"

## Prompt

Načtený ze souborů — ty jsou zdroj pravdy.


In [ ]:
system_prompt = (ROOT / "prompts" / "system.txt").read_text(encoding="utf-8")
user_template = (ROOT / "prompts" / "user_template.txt").read_text(encoding="utf-8")

print("=== system.txt (prvních 800 znaků) ===")
print(system_prompt[:800])
print("…\n")
print("=== user_template.txt (celá šablona) ===")
print(user_template)

## Offline

Bez cloud API: uložené `actual.json` přes harness. Live: API klíč a běh bez `--offline`.


In [ ]:
from konzultant.inputs import load_parcel, build_messages, load_schema
from konzultant.harness import load_recorded_actual, strip_meta, CASES_DIR
from konzultant.domain import CORE_AXES
from konzultant.checks import run_checks

schema = load_schema()

def run_offline(case_id: str):
    """Offline: load + verify recorded actual."""
    case_dir = CASES_DIR / case_id
    inp = case_dir / "input.json"
    if not inp.exists():
        inp = case_dir / "input.txt"
    parcel = load_parcel(str(inp))
    data = load_recorded_actual(case_id)  # schema + normalizace + brány
    body = strip_meta(data)
    expected = json.loads((case_dir / "expected.json").read_text(encoding="utf-8"))
    check_errs = run_checks(expected.get("kontroly") or {}, data)
    meta = data.get("_meta") or {}
    return {
        "case_id": case_id,
        "parcel_id": parcel.get("id") or case_id,
        "messages_preview": [m["role"] for m in build_messages(parcel)],
        "schema_ok": True,
        "schema_errs": [],
        "check_errs": check_errs,
        "meta": meta,
        "vyhodnoceni": body,
    }

def show(result: dict):
    v = result["vyhodnoceni"]
    print(f"\n######## {result['case_id']} ########")
    print(f"_meta.zdroj = {result['meta'].get('zdroj')}  |  model = {result['meta'].get('model')}")
    print(f"schema_ok={result['schema_ok']}  check_errs={result['check_errs'] or '[]'}")
    print(f"doporuceni = {v['doporuceni']}")
    print(f"skore_zajmu = {v['skore_zajmu']}  jistota = {v['jistota']['skore']}")
    print(f"hlavni_potencial = {v['vyuzitelnost']['hlavni_potencial']}")
    print("rizika:")
    for axis in CORE_AXES:
        r = v["rizika"][axis]
        print(f"  - {axis}: {r['uroven']} — {r['duvod'][:100]}")
    print("\n--- JSON (zkráceně shrnuti + doporuceni) ---")
    print(json.dumps({
        "shrnuti": v["shrnuti"],
        "doporuceni": v["doporuceni"],
        "skore_zajmu": v["skore_zajmu"],
        "skore_slozky": v["skore_slozky"],
        "rizika": {k: v["rizika"][k]["uroven"] for k in CORE_AXES},
        "jistota": v["jistota"],
        "overit_rucne": v["overit_rucne"],
    }, ensure_ascii=False, indent=2))


## Happy path — `01-stavebni-happy`

Parcela: proluka BH v Brně (`data/parcels/01-stavebni-happy.json`).

In [ ]:
happy = run_offline("01-stavebni-happy")
show(happy)
print("\n--- celý vyhodnocovací JSON ---")
print(json.dumps(happy["vyhodnoceni"], ensure_ascii=False, indent=2))

## Tvrdý case — `02-zaplava`

Aktivní zóna záplav + nezastavitelný ÚP (`data/parcels/02-zaplava.json`).

In [ ]:
hard = run_offline("02-zaplava")
show(hard)
print("\n--- celý vyhodnocovací JSON ---")
print(json.dumps(hard["vyhodnoceni"], ensure_ascii=False, indent=2))

## Volitelně: stejné přes CLI `evaluate.py`

Následující buňka volá skutečný skript (offline). Vyžaduje, aby běžela z kořene balíčku.

In [ ]:
import subprocess

for case in ("01-stavebni-happy", "02-zaplava"):
    cmd = ["python3", "evaluate.py", "--offline", f"tests/cases/{case}/input.json"]
    print("\n$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
    print("stderr:", proc.stderr.strip())
    print("exit:", proc.returncode)
    data = json.loads(proc.stdout)
    print(f"doporuceni={data['doporuceni']} skore={data['skore_zajmu']} zdroj={data.get('_meta', {}).get('zdroj')}")

## Kam dál

- Iterace promptu: [`notes/iterace.md`](../notes/iterace.md)
- Reflexe limitů: [`notes/reflexe.md`](../notes/reflexe.md)
- Offline sada: `python3 evaluate.py --run-all-tests`
- Live (až bude API klíč): `python3 evaluate.py --eval-live --save-actual`